In [1]:
import numpy as np
import os
from skimage import io
from skimage.transform import resize


# Images sizes and resize

### set input directories

In [5]:
input_landslide_images='./Bijie-landslide-dataset/landslide/image/'
input_landslide_dem='./Bijie-landslide-dataset/landslide/dem/'
input_landslide_dem_resized='./Bijie-landslide-dataset/landslide/dem_resized/'
input_landslide_mask='./Bijie-landslide-dataset/landslide/mask/'
input_landslide_images_crop='./Bijie-landslide-dataset/landslide/image_crop/'
input_landslide_dem_resized_crop='./Bijie-landslide-dataset/landslide/dem_resized_crop/'
input_landslide_mask_crop='./Bijie-landslide-dataset/landslide/mask_crop/'


input_nolandslide_images='./Bijie-landslide-dataset/non-landslide/image/'
input_nolandslide_dem='./Bijie-landslide-dataset/non-landslide/dem/'
input_nolandslide_dem_resized='./Bijie-landslide-dataset/non-landslide/dem_resized'
input_nolandslide_images_crop='./Bijie-landslide-dataset/non-landslide/image_crop/'
input_nolandslide_dem_resized_crop='./Bijie-landslide-dataset/non-landslide/dem_resized_crop'


Useful functions

In [16]:
# Useful function
def creation_folder(name):
    if not os.path.exists(name):
        os.makedirs(name)
        print('creation of ',name)


### resize DEM to initial images

In [17]:
def resize_dem(input_dem,input_image,output_folder):
    """
    Resize dem images to initial images and save to output_folder
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print('creation of ',output_folder)
    names=os.listdir(input_dem)
    
    for name in names:
        s_ima=io.imread('%s/%s'%(input_image,name)).shape
        dem=io.imread('%s/%s'%(input_dem,name))
        dem_resized = resize(dem, s_ima,anti_aliasing=True)
        name_save='%s/%s'%(output_folder,name)
        data=np.clip(dem_resized*255,0,255).astype(np.uint8)
        io.imsave(name_save,data)
    print('converted %d files'%len(names))

In [ ]:
resize_dem(input_landslide_dem,input_landslide_images,input_landslide_dem_resized)

In [6]:
%%capture --no-display
resize_dem(input_nolandslide_dem,input_nolandslide_images,input_nolandslide_dem_resized)

### check consistency with image tiles and get min,max sizes

In [8]:
def check_size_consistency1(input_images,input_dsm,input_mask):
    names=os.listdir(input_dsm)
    max_lig = 0
    max_col = 0
    min_lig=9999999
    min_col=9999999
    for name in names:
        s_ima=io.imread('%s/%s'%(input_images,name)).shape
        s_dsm=io.imread('%s/%s'%(input_dsm,name)).shape
        s_mas=io.imread('%s/%s'%(input_mask,name)).shape
        condition = ((s_ima[1]==s_dsm[1]) and (s_ima[0]==s_dsm[0])and  (s_ima[1]==s_mas[1]) and (s_ima[0]==s_mas[0]))
        if condition is False:
            print('err size for',name)
            print('size mask : ',s_mas)
            print('size dsm : ',s_dsm)
            print('size image : ',s_ima)
        if s_ima[0] > max_lig:
            max_lig=s_ima[0]
        if s_ima[1] > max_col:
            max_col=s_ima[1]
        if s_ima[0] < min_lig:
            min_lig=s_ima[0]
        if s_ima[1] < min_col:
            min_col=s_ima[1]
       
            
    print('checked ',len(names),' images')
    return min_lig,min_col,max_lig,max_col
        
def check_size_consistency2(input_images,input_dsm):
    names=os.listdir(input_dsm)
    max_lig = 0
    max_col = 0
    min_lig=9999999
    min_col=9999999
    for name in names:
        s_ima=io.imread('%s/%s'%(input_images,name)).shape
        s_dsm=io.imread('%s/%s'%(input_dsm,name)).shape
        condition = ((s_ima[1]==s_dsm[1]) and (s_ima[0]==s_dsm[0]))
        if condition is False:
            print('err size for',name)
            print('size dsm : ',s_dsm)
            print('size image : ',s_ima)
        if s_ima[0] > max_lig:
            max_lig=s_ima[0]
        if s_ima[1] > max_col:
            max_col=s_ima[1]
        if s_ima[0] < min_lig:
            min_lig=s_ima[0]
        if s_ima[1] < min_col:
            min_col=s_ima[1]

    print('checked ',len(names),' images')
    return min_lig,min_col,max_lig,max_col


In [8]:
min_lig,min_col,max_lig,max_col=check_size_consistency1(input_landslide_images,input_landslide_dem_resized,input_landslide_mask)

checked  770  images


In [9]:
min_lig,min_col,max_lig,max_col=check_size_consistency2(input_nolandslide_images,input_nolandslide_dem_resized)

checked  2003  images


### crop large images

In [9]:
def sliding_window(image, stepSize, windowSize):
  for y in range(0, image.shape[0], stepSize):
    for x in range(0, image.shape[1], stepSize):
      yield (x, y, image[y:y + windowSize[1], x:x + windowSize[0]])

def crop_image(name_im,dest_name,windowsize,overlap):
    """
    crop an image (png or jpg) with overlap and a given sliding window
    save results in dest_name

    """
    creation_folder(dest_name)
    im=io.imread(name_im)
    windows=sliding_window(im, windowsize-overlap, (windowsize,windowsize))
    
    # generic image name and extension
    name=os.path.splitext(os.path.split(name_im)[1])[0]
    ext=os.path.splitext(os.path.split(name_im)[1])[1]
    num_im=0
    for window in windows:
        name_save='%s/%s_%.4d%s'%(dest_name,name,num_im,ext)
        io.imsave(name_save,window[2])
        num_im+=1


### crop all small images in a given folder

In [10]:
def crop_images(input_folder,output_folder,window_size,overlap):
    """
    put all images in a given folder
    """
    names=os.listdir(input_folder)
    creation_folder(output_folder)

    for name in names:
        s_ima=io.imread('%s/%s'%(input_folder,name)).shape
        if (s_ima[0]> window_size) or (s_ima[1]> window_size):
            crop_image('%s/%s'%(input_folder,name),output_folder,window_size,overlap)
            #print('image',name,'cropped (size =',s_ima,')')
        else:
            commande='cp %s/%s %s'%(input_folder,name,output_folder)
            #print('image',name,'copied (size =',s_ima,')')
            os.system(commande)


In [14]:
%%capture --no-display
crop_images(input_landslide_images,input_landslide_images_crop,256,0)

In [15]:
%%capture --no-display
crop_images(input_landslide_dem_resized,input_landslide_dem_resized_crop,256,0)

In [16]:
%%capture --no-display
crop_images(input_landslide_mask,input_landslide_mask_crop,256,0)

In [17]:
%%capture --no-display
crop_images(input_nolandslide_images,input_nolandslide_images_crop,256,0)

In [18]:
%%capture --no-display
crop_images(input_nolandslide_dem_resized,input_nolandslide_dem_resized_crop,256,0)


### Homogenize images (size 256 x 256)


In [11]:
#folders input
folder_images='./Bijie-landslide-dataset/landslide/image_crop/'
folder_dem='./Bijie-landslide-dataset/landslide/dem_resized_crop/'
folder_mask='./Bijie-landslide-dataset/landslide/mask_crop/'

folder_images_nolandslide='./Bijie-landslide-dataset/non-landslide/image_crop/'
folder_dem_nolandslide='./Bijie-landslide-dataset/non-landslide/dem_resized_crop'
# sizes
size_window=256
size_min=100

#### Resize  images

In [12]:
def resize_im(input_folder,size_new,output_folder,size_min):
    """
    Resize dem images to a given shape
    """
    creation_folder(output_folder)
    names=os.listdir(input_folder)
    
    for name in names:
        ima=io.imread('%s/%s'%(input_folder,name))
        s_ima=ima.shape
        if (s_ima[0]> size_min) and (s_ima[1]> size_min):
            ima_resized = resize(ima, size_new,anti_aliasing=True)
            name_save='%s/%s'%(output_folder,name)
            data=np.clip(ima_resized*255,0,255).astype(np.uint8)
            io.imsave(name_save,data)
    print('converted %d files'%len(names))


In [207]:
output_images='./Bijie-landslide-dataset/landslide/images_256_resized/'
resize_im(folder_images,(256,256),output_images,size_min)

creation of  ./Bijie-landslide-dataset/landslide/images_256_resized/


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_16136/1442221696.py:15: UserWarning: ./Bijie-landslide-dataset/landslide/images_256_resized//wn021_0005.png is a low contrast image
  io.imsave(name_save,data)


converted 1561 files


In [ ]:
%%capture --no-display
output_images='./Bijie-landslide-dataset/landslide/dem_256_resized/'
resize_im(folder_dem,(256,256),output_images,size_min)
output_images='./Bijie-landslide-dataset/landslide/mask_256_resized/'
resize_im(folder_mask,(256,256),output_images,size_min)

In [209]:
%%capture --no-display
output_images='./Bijie-landslide-dataset/non-landslide/images_256_resized/'
resize_im(folder_images_nolandslide,(256,256),output_images,size_min)

output_images='./Bijie-landslide-dataset/non-landslide/dem_256_resized'
resize_im(folder_dem_nolandslide,(256,256),output_images,size_min)


#### Pad images with 4 different padding techniques

In [13]:
# information for padding
def get_pad_info(im_shape,size_window):
    pad_lig=size_window-im_shape[0]
    pad_col=size_window-im_shape[1]
    
    half_lig=pad_lig//2
    half_col=pad_col//2
    if pad_lig%2==0:
        extra_top=half_lig
        extra_bottom=half_lig
    else:
        extra_top=half_lig+1
        extra_bottom=half_lig
        
    
    if pad_col%2==0:
        extra_left=half_col
        extra_right=half_col
    else:
        extra_left=half_col+1
        extra_right=half_col
    if len(im_shape)==2:
        pad_size=(extra_top, extra_bottom), (extra_left, extra_right)
    else:
        pad_size=((extra_top, extra_bottom), (extra_left, extra_right), (0, 0))
    return pad_size


In [14]:
def pad_images(input_folder,size_window,size_min,output_folder,mode='symmetric'):
    
    names=os.listdir(input_folder)
    nb_pad=0
    creation_folder(output_folder)

    for name in names:
        ima=io.imread('%s/%s'%(input_folder,name))
        s_ima=ima.shape
        if (s_ima[0]> size_min) and (s_ima[1]> size_min):
            padinfo=get_pad_info(s_ima,size_window)
            imap=np.pad(ima, padinfo, mode=mode)
            name_save='%s/%s'%(output_folder,name)
            io.imsave(name_save,imap)
            nb_pad+=1
    print('put %d images over %d'%(nb_pad,len(names)))
            


In [19]:
modes = ('symmetric','linear_ramp','reflect','mean')
modes = ('constant',)
folder_root='./Bijie-landslide-dataset/landslide/images'
for mod in modes:
    out_folder_ima_la = '%s_%d_%s'%(folder_root,size_window,mod)
    pad_images(folder_images,size_window,size_min,out_folder_ima_la,mode=mod)


creation of  ./Bijie-landslide-dataset/landslide/images_256_constant
put 968 images over 1561


In [20]:
modes = ('symmetric','linear_ramp','reflect','mean')
modes = ('constant',)
folder_root='./Bijie-landslide-dataset/landslide/dem'
for mod in modes:
    out_folder_ima_la = '%s_%d_%s'%(folder_root,size_window,mod)
    pad_images(folder_dem,size_window,size_min,out_folder_ima_la,mode=mod)


creation of  ./Bijie-landslide-dataset/landslide/dem_256_constant


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/dem_256_constant/ny050_0018.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/dem_256_constant/ny050_0012.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/dem_256_constant/ny050_0017.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/dem_256_constant/ny050_0016.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/dem_256_c

put 968 images over 1561


In [21]:
modes = ('symmetric','linear_ramp','reflect','mean')
modes = ('constant',)
folder_root='./Bijie-landslide-dataset/landslide/mask'
for mod in modes:
    out_folder_ima_la = '%s_%d_%s'%(folder_root,size_window,mod)
    pad_images(folder_mask,size_window,size_min,out_folder_ima_la,mode=mod)


creation of  ./Bijie-landslide-dataset/landslide/mask_256_constant


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/wn046_0001.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/ny058_0003.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/hz042.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/hz095.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_consta

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/ny050_0023.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/ny050_0009.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/wn021_0005.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/zj058_0001.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/hz058.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/js019.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/qxg028.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/qx079.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/qx078.

put 968 images over 1561


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/landslide/mask_256_constant/df030_0003.png is a low contrast image
  io.imsave(name_save,imap)


In [22]:
folder_images_nolandslide='./Bijie-landslide-dataset/non-landslide/image_crop/'
folder_dem_nolandslide='./Bijie-landslide-dataset/non-landslide/dem_resized_crop'
modes = ('symmetric','linear_ramp','reflect','mean')
modes = ('constant',)
folder_root='./Bijie-landslide-dataset/non-landslide/images'
for mod in modes:
    out_folder_ima_la = '%s_%d_%s'%(folder_root,size_window,mod)
    pad_images(folder_images_nolandslide,size_window,size_min,out_folder_ima_la,mode=mod)



creation of  ./Bijie-landslide-dataset/non-landslide/images_256_constant


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/non-landslide/images_256_constant/fyb1538_0000.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/non-landslide/images_256_constant/dnypl05059_0000.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/non-landslide/images_256_constant/zb3dzj003_0003.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/non-landslide/images_256_constant/zb3dzj003_0001.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./

put 3855 images over 6257


In [23]:
folder_root='./Bijie-landslide-dataset/non-landslide/dem'
modes = ('symmetric','linear_ramp','reflect','mean')
modes = ('constant',)
for mod in modes:
    out_folder_ima_la = '%s_%d_%s'%(folder_root,size_window,mod)
    pad_images(folder_dem_nolandslide,size_window,size_min,out_folder_ima_la,mode=mod)


creation of  ./Bijie-landslide-dataset/non-landslide/dem_256_constant


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_256_constant/zb3dzj001_0001.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_256_constant/zb3dzj001_0000.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_256_constant/zb3dzj001_0002.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_256_constant/zb3dzj001_0003.png is a low contrast image
  io.imsave(name_save,imap)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10163/770539143.py:14: UserWarning: ./Bijie-lands

put 3855 images over 6257


Creation of masks of zeros for non-landslide

In [24]:
im_zeros=np.zeros((256,256))
print('im_zeros.shape',im_zeros.shape)

im_zeros.shape (256, 256)


In [26]:
%%capture --no-display
folder_root='./Bijie-landslide-dataset/non-landslide/mask'
modes = ('symmetric','linear_ramp','reflect','mean')
modes = ('constant',)
initial = './Bijie-landslide-dataset/non-landslide/images_256_symmetric/'
names=os.listdir(initial)

for mod in modes:
    out_folder_ima_la = '%s_%d_%s'%(folder_root,size_window,mod)
    creation_folder(out_folder_ima_la)
    for name in names:
        name_save='%s/%s'%(out_folder_ima_la,name)
        io.imsave(name_save,im_zeros)



### check consistency of folders

In [194]:
folder_root='./Bijie-landslide-dataset/non-landslide/'
modes = ('symmetric','linear_ramp','reflect','mean')
for mod in modes:
    input_landslide_images='%s/images_256_%s'%(folder_root,mod)
    input_landslide_dem='%s/dem_256_%s'%(folder_root,mod)
    input_landslide_mask='%s/mask_256_%s'%(folder_root,mod)
    check_size_consistency1(input_landslide_images,input_landslide_dem,input_landslide_mask)

checked  3855  images
checked  3855  images
checked  3855  images
checked  3855  images


In [195]:
folder_root='./Bijie-landslide-dataset/landslide/'
modes = ('symmetric','linear_ramp','reflect','mean')
for mod in modes:
    input_landslide_images='%s/images_256_%s'%(folder_root,mod)
    input_landslide_dem='%s/dem_256_%s'%(folder_root,mod)
    input_landslide_mask='%s/mask_256_%s'%(folder_root,mod)
    check_size_consistency1(input_landslide_images,input_landslide_dem,input_landslide_mask)

checked  968  images
checked  968  images
checked  968  images
checked  968  images
